# Allscripts SCM Episode Event Hydration

Link SCM episodes to related conditions, visits, measurements, and drug exposures.

## Episode event field concepts
- 1147127: `condition_occurrence.condition_occurrence_id`
- 1147126: `visit_occurrence.visit_occurrence_id`
- 1147130: `measurement.measurement_id`
- 1147132: `drug_exposure.drug_exposure_id`


In [ ]:
%sql
TRUNCATE TABLE _exponent.omop_scm.episode_event;


In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW episode_event_candidate AS

SELECT
  e.episode_id,
  co.condition_occurrence_id AS event_id,
  1147127 AS episode_event_field_concept_id
FROM _exponent.omop_scm.episode e
JOIN _exponent.omop_scm.condition_occurrence co
  ON co.person_id = e.person_id
 AND co.condition_concept_id = e.episode_object_concept_id
WHERE co.condition_start_date >= e.episode_start_date

UNION ALL

SELECT
  e.episode_id,
  vo.visit_occurrence_id AS event_id,
  1147126 AS episode_event_field_concept_id
FROM _exponent.omop_scm.episode e
JOIN _exponent.omop_scm.visit_occurrence vo
  ON vo.person_id = e.person_id
WHERE vo.visit_start_date >= e.episode_start_date

UNION ALL

SELECT
  e.episode_id,
  m.measurement_id AS event_id,
  1147130 AS episode_event_field_concept_id
FROM _exponent.omop_scm.episode e
JOIN _exponent.omop_scm.measurement m
  ON m.person_id = e.person_id
WHERE m.measurement_date >= e.episode_start_date

UNION ALL

SELECT
  e.episode_id,
  de.drug_exposure_id AS event_id,
  1147132 AS episode_event_field_concept_id
FROM _exponent.omop_scm.episode e
JOIN _exponent.omop_scm.drug_exposure de
  ON de.person_id = e.person_id
WHERE de.drug_exposure_start_date >= e.episode_start_date;

In [ ]:
%sql
INSERT INTO _exponent.omop_scm.episode_event (
  episode_id,
  event_id,
  episode_event_field_concept_id
)
SELECT DISTINCT
  episode_id,
  event_id,
  episode_event_field_concept_id
FROM episode_event_candidate;

In [ ]:
%sql
SELECT
  episode_event_field_concept_id,
  CASE episode_event_field_concept_id
    WHEN 1147127 THEN 'condition_occurrence'
    WHEN 1147126 THEN 'visit_occurrence'
    WHEN 1147130 THEN 'measurement'
    WHEN 1147132 THEN 'drug_exposure'
    ELSE 'other'
  END AS event_type,
  COUNT(*) AS event_count
FROM _exponent.omop_scm.episode_event
GROUP BY episode_event_field_concept_id
ORDER BY event_count DESC;